In [24]:
from pathlib import Path
import sys

CVS_DIR = Path.cwd().resolve()
if str(CVS_DIR) not in sys.path:
    sys.path.insert(0, str(CVS_DIR))

from cv_utils import build_folds_by_method

In [25]:
from cv_utils import build_folds_by_method 
import pandas as pd
import numpy as np
import geopandas as gpd

In [26]:
import importlib
import cv_plot_utils
import randomCV
import spatialCV
import cv_utils

importlib.reload(cv_plot_utils)
importlib.reload(randomCV)
importlib.reload(spatialCV)
importlib.reload(cv_utils)

from cv_utils import build_folds_by_method


In [27]:
pd.set_option('display.max_columns', None)

In [ ]:
dataframe = pd.read_csv("../Data/unitask_data/dataset.csv")

# Determining features and target columns
columns = dataframe.columns.to_list()
soil_properties = ['pH', 'CE', 'CEC', 'OM', 'A%', 'L%', 'S%']
cols_to_drop = soil_properties + ['X', 'Y']
covariates = [col for col in columns if col not in cols_to_drop]
covariates

['0',
 '1',
 '2',
 '3',
 '4',
 '5',
 '6',
 '7',
 '8',
 '9',
 '10',
 '11',
 '12',
 '13',
 '14',
 '15',
 '16',
 '17',
 '18',
 '19',
 '20',
 '21',
 '22',
 '23',
 '24',
 '25',
 '26',
 '27',
 '28',
 '29',
 '30',
 '31',
 '32',
 '33',
 '34',
 '35',
 '36',
 '37']

In [29]:
covariates = ['0', '2', '3', '20', '27' , '33']

gdf = gpd.GeoDataFrame(
    dataframe,
    geometry=gpd.points_from_xy(dataframe['X'], dataframe['Y']),
    crs='EPSG:4326'
)

gdf = gdf.to_crs('EPSG:32631')

# Keep expected 47-column layout: covariates + soil properties + X + Y
ordered_cols = covariates + soil_properties + ['X', 'Y']

dataframe_proj = dataframe.copy()
dataframe_proj['X'] = gdf.geometry.x.values
dataframe_proj['Y'] = gdf.geometry.y.values

points = dataframe_proj[ordered_cols].to_numpy()
print("Points shape:", points.shape)

Points shape: (2156, 15)


## For all Algeria 

In [30]:
N_FOLDS = 5
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [31]:

block_thresholds_km = [20, 30, 50, 100, 5, 10]
sp1_thresholds_km = [20, 30, 50, 100, 5, 10]
sp2_thresholds_km = [20, 30, 50, 100, 5, 10]


block_thresholds_m = [v * 1000 for v in block_thresholds_km]
sp1_thresholds_m = [v * 1000 for v in sp1_thresholds_km]
sp2_thresholds_m = [v * 1000 for v in sp2_thresholds_km]

n_cov = len(covariates)        
n_tar = len(soil_properties) 
print(n_cov, n_tar) 

env = points[:, :n_cov]                
tar = points[:, n_cov:n_cov + n_tar]  

folds_by_method = build_folds_by_method(
    region = 'all',
    points = points,
    env = env,
    tar = tar,
    n_folds=N_FOLDS,
    block_sp_threshold=block_thresholds_m,
    sp1_threshold=sp1_thresholds_m,
    sp_threshold=sp2_thresholds_m,
    random_repeats=1
)

print(sorted(folds_by_method.keys()))

6 7
D:\DSM_MODULE\CVs\cv_folds\all\random
D:\DSM_MODULE\CVs\cv_folds\all\sp1
D:\DSM_MODULE\CVs\cv_folds\all\sp2
D:\DSM_MODULE\CVs\cv_folds\all\sp
Building folds by method in D:\DSM_MODULE\CVs\cv_folds\all...
  - Random CV: 1 repeats, k=5
Generating random-CV folds.
Checking for existing CSV: D:\DSM_MODULE\CVs\cv_folds\all\random\randomCV_rep0_k5.csv
  - Block CV: k=5, thresholds=[20000, 30000, 50000, 100000, 5000, 10000]
    - Threshold: 20000
Generating block-CV folds.
    - Threshold: 30000
Generating block-CV folds.
    - Threshold: 50000
Generating block-CV folds.
    - Threshold: 100000
Generating block-CV folds.
    - Threshold: 5000
Generating block-CV folds.
    - Threshold: 10000
Generating block-CV folds.
  - SP1 CV: k=5, thresholds=[20000, 30000, 50000, 100000, 5000, 10000]
    - Threshold: 20000
    - Threshold: 30000
    - Threshold: 50000
    - Threshold: 100000
    - Threshold: 5000
    - Threshold: 10000
  - SP2 CV: k=5
  - SP CV: k=5, thresholds=[20000, 30000, 50000, 1

In [32]:
# Quick sanity check: thresholds are in meters in EPSG:32631
x_span = points[:, -2].max() - points[:, -2].min()
y_span = points[:, -1].max() - points[:, -1].min()

print(f"x_span = {x_span:.1f} m")
print(f"y_span = {y_span:.1f} m")

for thr in block_thresholds_m:
    x_blocks = int(x_span / thr) + 1
    y_blocks = int(y_span / thr) + 1
    total_blocks = x_blocks * y_blocks
    print(f"thr={thr:>6}: x_blocks={x_blocks:>6}, y_blocks={y_blocks:>6}, grid_cells={total_blocks:>12}")

x_span = 1116538.4 m
y_span = 1139701.2 m
thr= 20000: x_blocks=    56, y_blocks=    57, grid_cells=        3192
thr= 30000: x_blocks=    38, y_blocks=    38, grid_cells=        1444
thr= 50000: x_blocks=    23, y_blocks=    23, grid_cells=         529
thr=100000: x_blocks=    12, y_blocks=    12, grid_cells=         144
thr=  5000: x_blocks=   224, y_blocks=   228, grid_cells=       51072
thr= 10000: x_blocks=   112, y_blocks=   114, grid_cells=       12768


In [33]:
env.shape, tar.shape

((2156, 6), (2156, 7))

In [34]:
import numpy as np

# 1. Any NaNs already in the covariate block?
nan_cols = np.isnan(points.astype(float)).sum(axis=0)
print("NaNs per covariate column:\n", dict(zip(covariates, nan_cols)))

# 2. Any constant (zero-variance) covariate columns?
col_min = points.astype(float).min(axis=0)
col_max = points.astype(float).max(axis=0)
constant_cols = [covariates[i] for i in range(len(covariates)) if col_max[i] == col_min[i]]
print("Constant columns (will cause /0 in array_norm):", constant_cols)

NaNs per covariate column:
 {'0': np.int64(0), '2': np.int64(0), '3': np.int64(0), '20': np.int64(0), '27': np.int64(0), '33': np.int64(0)}
Constant columns (will cause /0 in array_norm): []


In [35]:
folds_by_method_by_region = {
    "all" : folds_by_method
}

# By region : North 

In [36]:
covariates = ['0', '2', '3', '20', '27' , '33']
latitude_boundary = 35.0
df_north = dataframe[dataframe['Y'] >  latitude_boundary]

gdf_north = gpd.GeoDataFrame(
    df_north,
    geometry=gpd.points_from_xy(df_north['X'], df_north['Y']),
    crs='EPSG:4326'
)

N_FOLDS = 5

gdf_north = gdf_north.to_crs('EPSG:32631')

# Keep expected 47-column layout: covariates + soil properties + X + Y
ordered_cols = covariates + soil_properties + ['X', 'Y']

dataframe_proj_north = df_north.copy()
dataframe_proj_north['X'] = gdf_north.geometry.x.values
dataframe_proj_north['Y'] = gdf_north.geometry.y.values

north_points = dataframe_proj_north[ordered_cols].to_numpy()
print("Points shape:", north_points.shape)

Points shape: (1449, 15)


In [37]:
block_thresholds_km = [20, 30, 50, 100, 5, 10]
sp1_thresholds_km = [20, 30, 50, 100, 5, 10]
sp2_thresholds_km = [20, 30, 50, 100, 5, 10]


block_thresholds_m = [v * 1000 for v in block_thresholds_km]
sp1_thresholds_m = [v * 1000 for v in sp1_thresholds_km]
sp2_thresholds_m = [v * 1000 for v in sp2_thresholds_km]

n_cov = len(covariates)        
n_tar = len(soil_properties) 
print(n_cov, n_tar) 

env = north_points[:, :n_cov]                
tar = north_points[:, n_cov:n_cov + n_tar]  

folds_by_method = build_folds_by_method(
    region = 'north',
    points = north_points,
    env = env,
    tar = tar,
    n_folds=N_FOLDS,
    block_sp_threshold=block_thresholds_m,
    sp1_threshold=sp1_thresholds_m,
    sp_threshold=sp2_thresholds_m,
    random_repeats=1
)

print(sorted(folds_by_method.keys()))

6 7
D:\DSM_MODULE\CVs\cv_folds\north\random
D:\DSM_MODULE\CVs\cv_folds\north\sp1
D:\DSM_MODULE\CVs\cv_folds\north\sp2
D:\DSM_MODULE\CVs\cv_folds\north\sp
Building folds by method in D:\DSM_MODULE\CVs\cv_folds\north...
  - Random CV: 1 repeats, k=5
Generating random-CV folds.
Checking for existing CSV: D:\DSM_MODULE\CVs\cv_folds\north\random\randomCV_rep0_k5.csv
  - Block CV: k=5, thresholds=[20000, 30000, 50000, 100000, 5000, 10000]
    - Threshold: 20000
Generating block-CV folds.
    - Threshold: 30000
Generating block-CV folds.
    - Threshold: 50000
Generating block-CV folds.
    - Threshold: 100000
Generating block-CV folds.
    - Threshold: 5000
Generating block-CV folds.
    - Threshold: 10000
Generating block-CV folds.
  - SP1 CV: k=5, thresholds=[20000, 30000, 50000, 100000, 5000, 10000]
    - Threshold: 20000
    - Threshold: 30000
    - Threshold: 50000
    - Threshold: 100000
    - Threshold: 5000
    - Threshold: 10000
  - SP2 CV: k=5
  - SP CV: k=5, thresholds=[20000, 300

In [38]:
folds_by_method_by_region[f'north_{N_FOLDS}'] = folds_by_method


# By region : south

In [39]:
df_south = dataframe[dataframe['Y'] <=  latitude_boundary]

gdf_south = gpd.GeoDataFrame(
    df_south,
    geometry=gpd.points_from_xy(df_south['X'], df_south['Y']),
    crs='EPSG:4326'
)

gdf_south = gdf_south.to_crs('EPSG:32631')

# Keep expected 47-column layout: covariates + soil properties + X + Y
ordered_cols = covariates + soil_properties + ['X', 'Y']

dataframe_proj_south = df_south.copy()
dataframe_proj_south['X'] = gdf_south.geometry.x.values
dataframe_proj_south['Y'] = gdf_south.geometry.y.values

south_points = dataframe_proj_south[ordered_cols].to_numpy()
print("Points shape:", south_points.shape)

Points shape: (707, 15)


In [40]:
block_thresholds_km = [20, 30, 50, 100, 5, 10]
sp1_thresholds_km = [20, 30, 50, 100, 5, 10]
sp2_thresholds_km = [20, 30, 50, 100, 5, 10]


block_thresholds_m = [v * 1000 for v in block_thresholds_km]
sp1_thresholds_m = [v * 1000 for v in sp1_thresholds_km]
sp2_thresholds_m = [v * 1000 for v in sp2_thresholds_km]

n_cov = len(covariates)        
n_tar = len(soil_properties) 
print(n_cov, n_tar) 

env = south_points[:, :n_cov]                
tar = south_points[:, n_cov:n_cov + n_tar]  

folds_by_method = build_folds_by_method(
    region = 'south',
    points = south_points,
    env = env,
    tar = tar,
    n_folds=N_FOLDS,
    block_sp_threshold=block_thresholds_m,
     sp1_threshold=sp1_thresholds_m,
    sp_threshold=sp2_thresholds_m,
    random_repeats=1
)

print(sorted(folds_by_method.keys()))

6 7
D:\DSM_MODULE\CVs\cv_folds\south\random
D:\DSM_MODULE\CVs\cv_folds\south\sp1
D:\DSM_MODULE\CVs\cv_folds\south\sp2
D:\DSM_MODULE\CVs\cv_folds\south\sp
Building folds by method in D:\DSM_MODULE\CVs\cv_folds\south...
  - Random CV: 1 repeats, k=5
Generating random-CV folds.
Checking for existing CSV: D:\DSM_MODULE\CVs\cv_folds\south\random\randomCV_rep0_k5.csv
  - Block CV: k=5, thresholds=[20000, 30000, 50000, 100000, 5000, 10000]
    - Threshold: 20000
Generating block-CV folds.
    - Threshold: 30000
Generating block-CV folds.
    - Threshold: 50000
Generating block-CV folds.
    - Threshold: 100000
Generating block-CV folds.
    - Threshold: 5000
Generating block-CV folds.
    - Threshold: 10000
Generating block-CV folds.
  - SP1 CV: k=5, thresholds=[20000, 30000, 50000, 100000, 5000, 10000]
    - Threshold: 20000
    - Threshold: 30000
    - Threshold: 50000
    - Threshold: 100000
    - Threshold: 5000
    - Threshold: 10000
  - SP2 CV: k=5
  - SP CV: k=5, thresholds=[20000, 300

In [41]:
folds_by_method_by_region[f'south_{N_FOLDS}'] = folds_by_method


In [42]:
folds_by_method_by_region

{'all': {'random': [[482, 0],
   [2137, 0],
   [10, 0],
   [734, 0],
   [635, 0],
   [321, 0],
   [237, 0],
   [1676, 0],
   [1337, 0],
   [794, 0],
   [200, 0],
   [1778, 0],
   [1137, 0],
   [1487, 0],
   [997, 0],
   [891, 0],
   [2084, 0],
   [1936, 0],
   [327, 0],
   [970, 0],
   [1836, 0],
   [102, 0],
   [21, 0],
   [955, 0],
   [419, 0],
   [658, 0],
   [983, 0],
   [703, 0],
   [2113, 0],
   [601, 0],
   [350, 0],
   [1726, 0],
   [739, 0],
   [283, 0],
   [1489, 0],
   [397, 0],
   [1507, 0],
   [1664, 0],
   [2091, 0],
   [753, 0],
   [1852, 0],
   [180, 0],
   [1859, 0],
   [1667, 0],
   [550, 0],
   [648, 0],
   [1774, 0],
   [446, 0],
   [957, 0],
   [449, 0],
   [1810, 0],
   [507, 0],
   [1226, 0],
   [767, 0],
   [520, 0],
   [1944, 0],
   [1490, 0],
   [859, 0],
   [466, 0],
   [393, 0],
   [772, 0],
   [1405, 0],
   [681, 0],
   [212, 0],
   [912, 0],
   [1191, 0],
   [2115, 0],
   [863, 0],
   [147, 0],
   [946, 0],
   [1380, 0],
   [2064, 0],
   [1047, 0],
   [475

In [43]:
import pickle

with open("cv_results.pkl", "wb") as f:
    pickle.dump(folds_by_method_by_region, f)

# Stats 

In [44]:
folds_by_method

{'random': [[553, 0],
  [30, 0],
  [705, 0],
  [582, 0],
  [676, 0],
  [136, 0],
  [222, 0],
  [100, 0],
  [199, 0],
  [478, 0],
  [502, 0],
  [679, 0],
  [451, 0],
  [240, 0],
  [247, 0],
  [110, 0],
  [207, 0],
  [503, 0],
  [83, 0],
  [637, 0],
  [449, 0],
  [202, 0],
  [650, 0],
  [522, 0],
  [275, 0],
  [221, 0],
  [271, 0],
  [570, 0],
  [443, 0],
  [373, 0],
  [683, 0],
  [432, 0],
  [627, 0],
  [406, 0],
  [506, 0],
  [665, 0],
  [391, 0],
  [560, 0],
  [458, 0],
  [52, 0],
  [470, 0],
  [74, 0],
  [258, 0],
  [113, 0],
  [21, 0],
  [232, 0],
  [31, 0],
  [141, 0],
  [360, 0],
  [32, 0],
  [547, 0],
  [533, 0],
  [633, 0],
  [568, 0],
  [245, 0],
  [238, 0],
  [320, 0],
  [164, 0],
  [630, 0],
  [471, 0],
  [501, 0],
  [248, 0],
  [385, 0],
  [63, 0],
  [231, 0],
  [126, 0],
  [536, 0],
  [453, 0],
  [134, 0],
  [526, 0],
  [659, 0],
  [532, 0],
  [378, 0],
  [642, 0],
  [401, 0],
  [201, 0],
  [235, 0],
  [325, 0],
  [229, 0],
  [596, 0],
  [192, 0],
  [632, 0],
  [598, 0],
  

In [45]:
CV_MEHODS= folds_by_method.keys()
CV_MEHODS

dict_keys(['random', 'block_k5_20000', 'block_k5_30000', 'block_k5_50000', 'block_k5_100000', 'block_k5_5000', 'block_k5_10000', 'sp1_k5_20000', 'sp1_k5_30000', 'sp1_k5_50000', 'sp1_k5_100000', 'sp1_k5_5000', 'sp1_k5_10000', 'sp2_k5', 'sp_k5_20000', 'sp_k5_30000', 'sp_k5_50000', 'sp_k5_100000', 'sp_k5_5000', 'sp_k5_10000'])

In [46]:
import numpy as np
import pandas as pd


CV_MEHODS = ['random', 'block_k5_20000', 'block_k5_30000', 'block_k5_50000', 'block_k5_10000', 'sp1_k5_20000', 'sp1_k5_30000', 'sp1_k5_50000', 'sp1_k5_10000', 'sp2_k5', 'sp_k5_20000', 'sp_k5_30000', 'sp_k5_50000', 'sp_k5_10000',
'block_k3_20000', 'block_k3_30000', 'block_k3_50000', 'block_k3_10000', 'sp1_k3_20000', 'sp1_k3_30000', 'sp1_k3_50000', 'sp1_k3_10000', 'sp_k3_20000', 'sp_k3_30000', 'sp_k3_50000', 'sp_k3_10000', 'sp2_k3']
for region, folds_by_method in folds_by_method_by_region.items():
    records = []
    for cv_method, fold_array in folds_by_method.items():
        if cv_method in CV_MEHODS:
            fold_array = np.asarray(fold_array)

            point_indices = fold_array[:, 0]
            fold_numbers = fold_array[:, 1].astype(int)

            for fold in range(0, max(fold_numbers) + 1):
                # Number of profiles assigned to this fold
                n_profiles = np.sum(fold_numbers == fold)

                records.append({
                    "region": region,
                    "cv_method": cv_method,
                    "fold": fold,
                    "n_profiles": n_profiles
                })

    # Convert to DataFrame
    df_counts = pd.DataFrame(records)

    # Pivot: folds as rows, CV methods as columns
    pivot_table = df_counts.pivot(
        index="fold",
        columns="cv_method",
        values="n_profiles"
    )

    # Add total number of profiles across the five folds
    pivot_table.loc["Total"] = pivot_table.sum(axis=0)

    # Save results
    pivot_table.to_csv(f"CV_profiles_per_fold_{region}.csv")
    print('region' , region)
    print(pivot_table)

region all
cv_method  block_k5_10000  block_k5_20000  block_k5_30000  block_k5_50000  \
fold                                                                        
0                     702             422             149             861   
1                     432             237             458             614   
2                     364             613             542             453   
3                     469             376             527             150   
4                     189             508             480              78   
Total                2156            2156            2156            2156   

cv_method  random  sp1_k5_10000  sp1_k5_20000  sp1_k5_30000  sp1_k5_50000  \
fold                                                                        
0             432           553           303           149           369   
1             431           290           793           647          1014   
2             431           435           563           129     